# Prepare Environment

We recommend using `uv` to run this script:
```shell
curl -LsSf https://astral.sh/uv/install.sh | sh
uv sync

## Prepare dataset

In this tutorial, we use the titanic dataset.
We split the dataset into three tables
- passengers_features: id, features...
- passengers_survived: id, survived (binary)
- is_adult_male: type of person (man or woman or child), is_adult_male (binary:=man is true)

In [ ]:
import seaborn as sns
import pandas as pd
import duckdb

# titanic dataset
titanic_data = sns.load_dataset('titanic')

# add id column and set it as primary key, and move it to the first column
titanic_data['id'] = range(1, len(titanic_data) + 1)
titanic_data = titanic_data[['id'] + [col for col in titanic_data.columns if col != 'id']]

# user_id, survived table
user_id_alive = titanic_data[['id', 'survived']]

# who, adult_male table
who_adult_male = titanic_data[['who', 'adult_male']].drop_duplicates()

# user_id, else table
user_id_else = titanic_data.drop(columns=['survived'])
user_id_else = user_id_else.drop(columns=['adult_male'])


# create a duckdb database in memory and add the tables
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE passengers_features AS SELECT * FROM user_id_else")
conn.execute("CREATE TABLE passengers_survived AS SELECT * FROM user_id_alive")
conn.execute("CREATE TABLE is_adult_male AS SELECT * FROM who_adult_male")
conn.commit()

# Let's protect privacy of passengers!

We are making the following query private

In [ ]:
sql = "SELECT survived, count(*) FROM passengers_survived GROUP BY survived"

## Setting privacy budget ε,δ

- ε represents the budget you can consume to analyze private data in the dataset
    - In bigger epsilon, you can analyze data many times but privacy leaks more
- δ represents the probability that infinite privacy leaks
    - We recommend setting δ <= 1/(n*log(n)) where n is the number of rows in the dataset

Accountant automatically accounts the budget you are consuming and raises an error when the budget is exhausted

In [ ]:
epsilon = 1.0
delta = 1e-4

from dpsql.accountant import RenyiAccountant

accountant = RenyiAccountant(epsilon, delta)

## Setup engine

In [ ]:
from dpsql.backend import DuckDBBackend
from dpsql.engine import Engine
from dpsql.validator import Validator


sql_backend = DuckDBBackend(conn)
validator = Validator()
engine = Engine(accountant, sql_backend, validator)

You need to register databases you want to analyze with the user id
- privacy is protected from the perspective of each user specified by the user id

In [ ]:
# Register databases into the engine
# Here, we want to protect privacy of each passenger , so we set the privacy unit column to "id" in the passengers_survived table
engine.register_database(privacy_unit_columns={"passengers_survived": "id"})


Now, you can query the databases.

Decide the parameters for query; you balance privacy and utility.

FYI: following is a intuition of the parameters
- contribution_bound: how many times a passenger contribute to aggregation (lower adds bias but consumes less privacy budget)
- min_frequency: how many numbers of passengers must include in aggregation (it satisfies minimum frequency rule)

In [ ]:
from dpsql.dp_params import DPParams

contribution_bound = 1
min_frequency = 10
epsilon_per_query = 0.5
delta_per_query = 5e-5
clipping_thresholds = [None] 

dpparams = DPParams(
    contribution_bound=contribution_bound, 
    min_frequency=min_frequency, 
    epsilon=epsilon_per_query, 
    delta=delta_per_query, 
    clipping_thresholds=clipping_thresholds
)

Finally, you are prepared to query!

In [ ]:
result_df = engine.execute_query(sql, dpparams)
result_df

You see the different result from the real one; this is the privacy

# Multiple Databases (tables) and Multiple Aggregations

Our DPSQL system supports subquery as the normal SQL systems.

Let's make the following query private

In [ ]:
sql = """
WITH combined_data AS (
    SELECT
        e.id,
        e.who,
        s.survived
    FROM
        passengers_features e
    JOIN
        passengers_survived s ON e.id = s.id
)
SELECT
    adult_male, COUNT(adult_male), SUM(survived)
FROM
    combined_data JOIN is_adult_male w ON combined_data.who = w.who
GROUP BY
    w.adult_male
LIMIT 100
"""

You need to add multiple tables in multiple databases as following

In [ ]:
# Register databases into the engine
# Here, we want to protect privacy of each passenger, so we set the privacy unit columns to "id" in both passengers_features and passengers_survived tables.
engine.register_database(privacy_unit_columns={"passengers_features": "id", "passengers_survived": "id"})

In [ ]:
titanic_data = conn.execute(sql).df()
titanic_data

You first rewrite the query to use nested CTEs (Common Table Expressions)

In [ ]:
sql = """
WITH combined_data_tmp AS (
    SELECT 
        e.id,
        e.who,
        s.survived
    FROM 
        passengers_features AS e
    JOIN 
        passengers_survived AS s ON e.id = s.id
), combined_data AS (
    SELECT
        c.id,
        c.who,
        c.survived,
        w.adult_male
    FROM
        combined_data_tmp AS c
    JOIN
        is_adult_male AS w ON c.who = w.who
)
SELECT
    adult_male, COUNT(adult_male), SUM(survived)
FROM
    combined_data
GROUP BY
    adult_male
"""


Decide the parameters for query; here, you have two aggregations.

Sum aggregation has a unique parameter clipping.
- clipping: value is clipped into the clipping value. survived takes 0 or 1 so you should set clipping as 1

In [ ]:
contribution_bound = 1
min_frequency = 10
epsilon_per_query = 0.5
delta_per_query = 5e-5

clipping = 1

dpparams = DPParams(
    contribution_bound=contribution_bound, 
    min_frequency=min_frequency, 
    epsilon=epsilon_per_query, 
    delta=delta_per_query, 
    clipping_thresholds=[None, [(0, clipping)]]
)

DONE! Let's query.

In [ ]:
result_df = engine.execute_query(sql, dpparams)
result_df

### Achieve Everything with SQL 
You can define the same contents in SQL instead of defining DPParams in Python.

In [ ]:
sql = """
WITH combined_data_tmp AS (
    SELECT 
        e.id,
        e.who,
        s.survived
    FROM 
        passengers_features AS e
    JOIN 
        passengers_survived AS s ON e.id = s.id
), combined_data AS (
    SELECT
        c.id,
        c.who,
        c.survived,
        w.adult_male
    FROM
        combined_data_tmp AS c
    JOIN
        is_adult_male AS w ON c.who = w.who
)
SELECT PRIVATE_QUERY OPTIONS (
    EPSILON = 0.5,
    DELTA = 5e-5,
    MIN_FREQUENCY = 10,
    CONTRIBUTION_BOUND = 1
)
    adult_male, COUNT(adult_male), SUM(survived, 0, 1)
FROM
    combined_data
GROUP BY
    adult_male
"""
result_df = engine.execute_query(sql, None)
result_df

If you run out of your budget, you cannot analyze anymore to protect privacy of passengers.

In [ ]:
for i in range(10):
    print(f"Query {i+1}")
    result_df = engine.execute_query(sql, None)

# Specify sigmas in DPParams

You can specify the sigmas and other noise parameters in DPParams instead of passing epsilon and delta.

In [ ]:
epsilon = 1.0
delta = 1e-4

accountant = RenyiAccountant(epsilon, delta)
sql_backend = DuckDBBackend(conn)
validator = Validator()
engine = Engine(accountant, sql_backend, validator)
engine.register_database(privacy_unit_columns={"passengers_survived": "id"})

In [ ]:
sql = "SELECT survived, count(*) FROM passengers_survived GROUP BY survived"

FYI: following is a intuition of the parameters
- tau: how many numbers of passengers should include in aggregation (tau >= min_frequency is required)
- sigma, sigma_for_thresholding: magnitude of noise; following relation is recommended
    - count: sigma = sigma_for_thresholding
    - sum: sigma = sigma_for_thresholding * clipping

In [ ]:
contribution_bound = 1
min_frequency = 10
tau = 100
sigma = 20
sigma_for_thresholding = sigma
clipping_thresholds = [None]

dpparams = DPParams(
    contribution_bound=contribution_bound, 
    min_frequency=min_frequency, 
    tau=tau, 
    sigma_for_thresholding=sigma_for_thresholding, 
    sigmas=[sigma], 
    clipping_thresholds=clipping_thresholds
)

In [ ]:
result_df = engine.execute_query(sql, dpparams)
result_df